In [ ]:
%matplotlib tk 
# to show the plot in pop-up windows
import tkinter
from tkinter import filedialog
from tkinter import filedialog, messagebox
import pandas as pd
import glob
import os
import h5py
import ipywidgets as widgets
from pandastable import Table # pip install pandastable
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from tkinter import ttk


def set_path(entry_field):
    path = filedialog.askdirectory() # get selected path
    if path:
        entry_field.delete(0, tkinter.END) # clean old route
        entry_field.insert(0, path) # put the new one in
        

def get_path():
    global pt
    folder_path = txt_path.get()
    print(f"Current route is: {folder_path}")
    # print(f"if this route exists: {os.path.exists(folder_path)}") # if shows False，the route is wrong
    records = []

    for file in glob.glob(f"{folder_path}"+"/*.nxspe"): # folder rout +*.nxspe
        name = os.path.basename(file).replace(".nxspe", "")
        if name.startswith("empty"):
            continue  # skip empty runs
        
        parts = name.split("_")
        
        sample = parts[0]
        temperature = float(parts[1])
        Ei = float(parts[2])
        scattering = parts[3]  # coh / inc
        
        records.append({
            "filepath": file,
            "sample": sample,
            "temperature": temperature,
            "Ei": Ei,
            "scattering": scattering
        })

    metadata = pd.DataFrame(records)
    if not metadata.empty:
        # show pandas dataframe on gui by using pandastable package
        pt.model.df = metadata
        pt.redraw()
        print("Table renewed")

# when double click  event trigger
def handle_row_click(event):
    """
    when double click trigger, start extract filepath and sample name
    """
    try:
        row_clicked = pt.get_row_clicked(event)
        
        if row_clicked < 0: return
        
        # obtain the gui selected route
        selected_row = pt.model.df.iloc[row_clicked]
        file_path = selected_row['filepath']
        
        print(f"\n[Loading file...] {selected_row['sample']}...")

        # execute
        data, energy, angles, ei, errors = load_nxspe_data(file_path)
        
        print(f"Succeed {selected_row['sample']}")
        print(f"data dimension: {data.shape} (Angles x Energy)")
        print(f"Ei: {ei[0] if isinstance(ei, np.ndarray) else ei}")

        plot_data(data, energy, angles, selected_row['sample'])

    except Exception as e:
        print(f"Fail: {e}")
        messagebox.showerror("Fail", str(e))

def load_nxspe_data(file_path):
    """
    Loading the selected nxspe data content
    """
    with h5py.File(file_path, 'r') as f:
        root_key = list(f.keys())[0] 
        print(f"Detect root key: {root_key}")

        data   = f[f'{root_key}/data/data'][()] 
        energy = f[f'{root_key}/data/energy'][()]
        angles = f[f'{root_key}/data/polar'][()]
        ei     = f[f'{root_key}/NXSPE_info/fixed_energy'][()]
        errors = f[f'{root_key}/data/error'][()]
        
        return data, energy, angles, ei, errors


def plot_data(data, energy, angles, title):
    """
    plotting the data (S(w,q) and w )
    note: the way to calculate the data can be potentially wrong
    since the array dimension of energy, angle and data, column and row number are not as expected
    """
    #pop-up window
    plot_ctrl = tkinter.Toplevel(root)
    plot_ctrl.title(f"Plot Control - {title}")
    plot_ctrl.geometry("300x150")

    q_list = angles.tolist()
    default_num = len(q_list) - 1 # 212

    tkinter.Label(plot_ctrl, text="Select Q value:").pack(pady=5)

    q_options = [f"{q:.4f} Å⁻¹" for q in q_list]
    w_dropdown = ttk.Combobox(plot_ctrl, values=q_options, state="readonly", width=25)
    w_dropdown.current(default_num) # default -> last one
    w_dropdown.pack(pady=5)

    def on_confirm():
        selected_index = w_dropdown.current()
        selected_q = q_list[selected_index]
        
        print(f"Current Index: {selected_index}")
        print(f"Corresponding Q: {selected_q:.4f} Å⁻¹")
        print("--- Framework ready, start plotting ---")
        
        fig, ax = plt.subplots(figsize=(8, 6))
        
        index = data[:, 0].shape[0] - default_num + 1 
        
        ax.plot(energy[:-1], data[index, :])
        ax.set_xlabel('w / meV')
        ax.set_ylabel('S(w, q)')
        ax.set_title(f'Q = {selected_q:.4f} in {title}')
        plt.show()

    # Confirm button -> trigger plotting
    btn_draw = tkinter.Button(plot_ctrl, text="Confirm & Plot", command=on_confirm, bg="white")
    btn_draw.pack(pady=10)
#========================================================
#                        GUI part
#=========================================================
root = tkinter.Tk()
root.title("QUENS GUI")
root.geometry("800x600") # window size

lbl_title = tkinter.Label(root, text="QUENS data analysis system", font=("Arial", 14))
lbl_title.pack(pady=50)

txt_path = tkinter.Entry(root, width=200)
txt_path.pack(pady=20)
# select path
btn_get_path = tkinter.Button(root, text="Select file",
                              command=lambda: set_path(txt_path))
btn_get_path.pack()
# execute get_path to obtain the metadata table
btn_run = tkinter.Button(root, text="File Selected", 
                         command=get_path, 
                         bg="white", height=2)
btn_run.pack(pady=20)

# gui table size
frame_table = tkinter.Frame(root)
frame_table.pack(fill='both', expand=True, padx=10, pady=10)

# show table
pt = Table(frame_table, dataframe=pd.DataFrame(), showtoolbar=True, showstatusbar=True)
pt.show()

pt.bind("<ButtonRelease-1>", handle_row_click) 
root.mainloop()

# if nothing shows please check the folder route, 
# the windows and wsl route format is different
# window route -> wsl = not ok
# wsl route -> wsl = ok

Current route is: 
